In [7]:
install.packages("BiocManager")
BiocManager::install(c("ensembldb", "AnnotationHub"))

Installing package into '/usr/local/lib/R/site-library'
(as 'lib' is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Warning message:
"package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'ensembldb' 'AnnotationHub'"
Old packages: 'cpp11', 'xtable'



In [8]:
library(ensembldb)
library(AnnotationHub)

ah <- AnnotationHub()
edb <- ah[["AH119325"]]

loading from cache



In [9]:
apoe_transcripts <- transcripts(edb, filter = ~ gene_id == "ENSG00000130203")
print(apoe_transcripts)

write.table(
  as.data.frame(apoe_transcripts),
  file = "APOE_transcripts.tsv",
  sep = "\t", row.names = FALSE, quote = FALSE
)

GRanges object with 5 ranges and 11 metadata columns:
                  seqnames            ranges strand |           tx_id
                     <Rle>         <IRanges>  <Rle> |     <character>
  ENST00000446996       19 44905791-44908944      + | ENST00000446996
  ENST00000485628       19 44905796-44907326      + | ENST00000485628
  ENST00000252486       19 44905796-44909393      + | ENST00000252486
  ENST00000434152       19 44905812-44909025      + | ENST00000434152
  ENST00000425718       19 44906360-44908954      + | ENST00000425718
                       tx_biotype tx_cds_seq_start tx_cds_seq_end
                      <character>        <integer>      <integer>
  ENST00000446996  protein_coding         44906625       44908944
  ENST00000485628 retained_intron             <NA>           <NA>
  ENST00000252486  protein_coding         44906625       44909250
  ENST00000434152  protein_coding         44905869       44909025
  ENST00000425718  protein_coding         44906625       449

In [10]:
apoe_proteins <- proteins(edb, filter = ~ gene_id == "ENSG00000130203")
print(apoe_proteins)

write.table(
  as.data.frame(apoe_proteins),
  file = "APOE_proteins.tsv",
  sep = "\t", row.names = FALSE, quote = FALSE
)

DataFrame with 4 rows and 4 columns
            tx_id      protein_id       protein_sequence         gene_id
      <character>     <character>            <character>     <character>
1 ENST00000446996 ENSP00000413135 MKVLWAALLVTFLAGCQAKV.. ENSG00000130203
2 ENST00000252486 ENSP00000252486 MKVLWAALLVTFLAGCQAKV.. ENSG00000130203
3 ENST00000434152 ENSP00000413653 MSSGASRKSWDPGNPWPPDW.. ENSG00000130203
4 ENST00000425718 ENSP00000410423 MKVLWAALLVTFLAGCQAKV.. ENSG00000130203


In [11]:
apoe_proteins_df <- as.data.frame(proteins(edb, filter = ~ gene_id == "ENSG00000130203"))

results_list <- list()

for (i in 1:nrow(apoe_proteins_df)) {
  prot_id  <- apoe_proteins_df$protein_id[i]
  prot_seq <- apoe_proteins_df$protein_sequence[i]
  prot_length <- nchar(prot_seq)

  aa_range <- IRanges(start = 1, end = prot_length)
  names(aa_range) <- prot_id

  genome_map <- proteinToGenome(aa_range, edb)

  df <- as.data.frame(genome_map[[prot_id]])
  df$protein_id <- prot_id
  results_list[[prot_id]] <- df
}

final_map <- do.call(rbind, results_list)
print(final_map)

write.table(
  final_map,
  file = "APOE_protein_to_genome_map.tsv",
  sep = "\t", row.names = FALSE, quote = FALSE
)

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000413135'. The returned genomic coordinates might thus not be correct for this protein."
0/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000413653'. The returned genomic coordinates might thus not be correct for this protein."
0/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000410423'. The returned genomic coordinates might thus not be correct for this protein."
0/1 OK



                  seqnames    start      end width strand      protein_id
ENSP00000413135.1       19 44906625 44906667    43      + ENSP00000413135
ENSP00000413135.2       19 44907760 44907952   193      + ENSP00000413135
ENSP00000413135.3       19 44908533 44908944   412      + ENSP00000413135
ENSP00000252486.1       19 44906625 44906667    43      + ENSP00000252486
ENSP00000252486.2       19 44907760 44907952   193      + ENSP00000252486
ENSP00000252486.3       19 44908533 44909247   715      + ENSP00000252486
ENSP00000413653.1       19 44905869 44905923    55      + ENSP00000413653
ENSP00000413653.2       19 44906602 44906667    66      + ENSP00000413653
ENSP00000413653.3       19 44907760 44907952   193      + ENSP00000413653
ENSP00000413653.4       19 44908533 44909025   493      + ENSP00000413653
ENSP00000410423.1       19 44906625 44906667    43      + ENSP00000410423
ENSP00000410423.2       19 44907760 44907952   193      + ENSP00000410423
ENSP00000410423.3       19 44908533 44